In [ ]:
!pip install emoji nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 1.7 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 2.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.4/803.4 kB 3.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [nltk]4/5 [nltk]]


In [1]:
import pandas as pd
import numpy as np
import re
import json
from sklearn.preprocessing import MultiLabelBinarizer
import emoji
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk

In [ ]:

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /home/tharun/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/tharun/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/tharun/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [2]:
# Load the CSV file
csv_file = 'project-1-at-2026-01-27-05-06-e12e4e5a.csv'
df = pd.read_csv(csv_file)
print("Data loaded. Shape:", df.shape)
print("Columns:", df.columns.tolist())

Data loaded. Shape: (229, 41)
Columns: ['annotation_id', 'annotator', 'created_at', 'favorite_count', 'followers_count', 'following_count', 'full_text', 'hashtags', 'id', 'is_blue_verified', 'is_news', 'is_political', 'is_quote', 'lang', 'lead_time', 'link', 'overall_sentiment', 'party_list', 'place', 'possibly_sensitive', 'quote_count', 'reply_count', 'retweet_count', 'sarcasm_check', 'screen_name', 'stance_admk', 'stance_bjp', 'stance_congress', 'stance_dmk', 'stance_ntk', 'stance_tvk', 'total_tweets_by_user', 'tweet_id', 'updated_at', 'urls', 'user_desc', 'user_id', 'user_location', 'user_name', 'user_verified', 'view_count']


In [3]:
# Explore the data
print(df.head())
print("\nUnique values in is_political:", df['is_political'].unique())
print("Unique values in party_list:", df['party_list'].unique()[:10])  # First 10
print("Stance columns:", [col for col in df.columns if 'stance_' in col])
print("Sample stance_admk:", df['stance_admk'].unique())

   annotation_id  annotator                   created_at  favorite_count  \
0             11          6  2026-01-20T14:22:59.630701Z             702   
1             12          6  2026-01-20T14:25:20.509030Z           60378   
2             13          6  2026-01-20T14:27:36.024833Z            1248   
3            104          3  2026-01-23T07:51:50.589081Z            3083   
4             15          6  2026-01-20T14:32:44.022103Z            1074   

   followers_count  following_count  \
0          1007659              883   
1          5519898                1   
2            35077              244   
3          4000735               91   
4             1993             2016   

                                           full_text hashtags  id  \
0  திருப்பூர் மாவட்டம் இடுவாய் கிராமத்தில், பொதும...       []   1   
1  Thank you Erode 🙏🏻 <a href="https://t.co/897AD...       []   2   
2  நடிகனுக்கு ஜால்ரா அடித்து வாங்கும் பதவி என் மய...       []   3   
3  Hon'ble PM Thiru. @NarendraMo

In [4]:
# Text preprocessing function
def preprocess_text(text):
    if pd.isna(text):
        return ""
    # Convert to lowercase
    text = text.lower()
    # Demojize emojis
    text = emoji.demojize(text)
    # Remove <a> elements
    text = re.sub(r'<a[^>]*>.*?</a>', '', text, flags=re.DOTALL)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stop words
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word.lower() not in stop_words]
    text = ' '.join(filtered_tokens)
    # Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply preprocessing
df['clean_text'] = df['full_text'].apply(preprocess_text)
print("Preprocessing done. Sample clean text:")
print(df[['full_text', 'clean_text']].head())

Preprocessing done. Sample clean text:
                                           full_text  \
0  திருப்பூர் மாவட்டம் இடுவாய் கிராமத்தில், பொதும...   
1  Thank you Erode 🙏🏻 <a href="https://t.co/897AD...   
2  நடிகனுக்கு ஜால்ரா அடித்து வாங்கும் பதவி என் மய...   
3  Hon'ble PM Thiru. @NarendraModi, Tamil Nadu is...   
4  நாம் தமிழர் பிள்ளைகள் சேலம் மாவட்டம் தரமான சம்...   

                                          clean_text  
0  திருப்பூர் மாவட்டம் இடுவாய் கிராமத்தில் , பொது...  
1       thank erode : folded_hands_light_skin_tone :  
2  நடிகனுக்கு ஜால்ரா அடித்து வாங்கும் பதவி என் மய...  
3  hon'ble pm thiru . @ narendramodi , tamil nadu...  
4  நாம் தமிழர் பிள்ளைகள் சேலம் மாவட்டம் தரமான சம்...  


In [5]:
df[['full_text', 'clean_text']].to_csv('review_clean_text.csv', index=False)

In [ ]:
# Create three CSV files for model training and testing

import datetime
suffix = datetime.datetime.now().strftime('%Y-%m-%d')

# First CSV: tweet_text, is_political
binary_df = df[['clean_text', 'is_political']].copy()
binary_df.to_csv(f'data/tweet_is_political_{suffix}.csv', index=False)

# Prepare parties_mentioned
def parse_party_list(party_str):
    if pd.isna(party_str) or party_str == '[]':
        return []
    try:
        # Try to parse as dict
        party_dict = json.loads(party_str.replace("'", '"'))  # In case single quotes
        if isinstance(party_dict, dict) and 'choices' in party_dict:
            return party_dict['choices']
        elif isinstance(party_dict, list):
            return party_dict
        else:
            # If it's a string, treat as single party
            return [party_str]
    except:
        # If parsing fails, assume it's a single party name
        return [party_str]

df['parties_mentioned'] = df['party_list'].apply(parse_party_list)

# Second CSV: tweet_text, Party, is_mentioned (76 tweets x 6 parties = 456 rows)
parties = ['ADMK', 'BJP', 'Congress', 'DMK', 'NTK', 'TVK']
party_mention_data = []
for idx, row in df.iterrows():
    if row['is_political'] == "Yes":
        for party in parties:
            is_mentioned = 1 if party in row['parties_mentioned'] else 0
            party_mention_data.append({
                'tweet_text': row['clean_text'],
                'Party': party,
                'is_mentioned': is_mentioned
            })
party_mention_df = pd.DataFrame(party_mention_data)
party_mention_df.to_csv(f'data/tweet_party_mention_{suffix}.csv', index=False)

# Prepare stance data
stance_columns = [col for col in df.columns if col.startswith('stance_')]
party_mapping = {
    'stance_admk': 'ADMK',
    'stance_bjp': 'BJP',
    'stance_congress': 'Congress',
    'stance_dmk': 'DMK',
    'stance_ntk': 'NTK',
    'stance_tvk': 'TVK'
}

stance_data = []
for idx, row in df.iterrows():
    for stance_col, party in party_mapping.items():
        stance = row[stance_col]
        if pd.notna(stance) and stance != '':
            stance_data.append({
                'tweet_id': row['tweet_id'],
                'clean_text': row['clean_text'],
                'party': party,
                'stance': stance
            })

stance_df = pd.DataFrame(stance_data)

# Third CSV: tweet_text, party, stance (one-hot encoded) - only for mentioned parties
unique_stances = stance_df['stance'].unique()
stance_encoded = pd.get_dummies(stance_df['stance'], prefix='stance')
stance_onehot_df = pd.concat([stance_df[['clean_text', 'party']], stance_encoded], axis=1)
stance_onehot_df.to_csv(f'data/tweet_party_stance_onehot_{suffix}.csv', index=False)

print("CSV files created in data/ folder:")
print(f"- data/tweet_is_political_{suffix}.csv")
print(f"- data/tweet_party_mention_{suffix}.csv")
print(f"- data/tweet_party_stance_onehot_{suffix}.csv")

CSV files created in data/ folder:
- data/tweet_is_political_2026-02-15.csv
- data/tweet_party_mention_2026-02-15.csv
- data/tweet_party_stance_onehot_2026-02-15.csv
